# Notebook B — Inference Only

Loads saved LoRA adapter from Kaggle Dataset and runs log-likelihood scoring on test set. No training — ~1 hour runtime.

In [ ]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow tqdm

In [ ]:
import os, ast
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

DATA_DIR    = Path("/kaggle/input/competitions/pixels-to-predictions")
ADAPTER_DIR = Path("/kaggle/input/datasets/anumathur/molvlm-finetuned-b/outputs/lora_adapter")
MODEL_ID    = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE      = 224
MAX_SEQ_LEN   = 2048
CHOICE_LABELS = "ABCDEFGH"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
print("Adapter files found:")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")


In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df["choices"] = test_df["choices"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
print(f"Test examples: {len(test_df):,}")


In [ ]:
def build_prompt(row):
    parts = ["<image>"]
    lecture = row.get("lecture", None)
    if pd.notna(lecture) and str(lecture).strip():
        parts.append(f"Context:\n{str(lecture).strip()}")
    hint = row.get("hint", None)
    if pd.notna(hint) and str(hint).strip():
        parts.append(f"Hint:\n{str(hint).strip()}")
    parts.append(f"Question: {row['question'].strip()}")
    parts.append("Choices:")
    for i, c in enumerate(row["choices"]):
        parts.append(f"  {CHOICE_LABELS[i]}. {c}")
    parts.append("Answer:")
    return "\n".join(parts)


In [ ]:
class ScienceQATestDataset:
    def __init__(self, df, data_dir, img_size=224):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
    def __len__(self): return len(self.df)
    def _load_image(self, rel_path):
        fixed = str(rel_path).replace("images/", "images/images/", 1)
        return (Image.open(self.data_dir / fixed)
                .convert("RGB")
                .resize((self.img_size, self.img_size), Image.BICUBIC))
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {"id": row["id"],
                "image": self._load_image(row["image_path"]),
                "prompt": build_prompt(row),
                "choices": row["choices"]}

test_ds = ScienceQATestDataset(test_df, DATA_DIR, img_size=IMG_SIZE)
sample = test_ds[0]
print(f"Dataset ready: {len(test_ds)} examples | Image size: {sample['image'].size}")


In [ ]:
processor = AutoProcessor.from_pretrained(str(ADAPTER_DIR))
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
print("Processor loaded.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
) if torch.cuda.is_available() else None

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
print(f"Base model loaded. Params: {sum(p.numel() for p in base_model.parameters()):,}")

model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model = model.merge_and_unload()
model.eval()
print("LoRA adapter merged and model set to eval mode.")


In [ ]:
@torch.inference_mode()
def score_choices_loglik(model, processor, image, base_prompt, choices):
    scores = []
    for i in range(len(choices)):
        full_prompt = base_prompt + " " + CHOICE_LABELS[i]
        inputs = processor(text=[full_prompt], images=[image],
                           return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
        labels = torch.full_like(inputs["input_ids"], -100)
        labels[:, -1] = inputs["input_ids"][:, -1]
        inputs["labels"] = labels
        outputs = model(**inputs)
        scores.append(-outputs.loss.item())
    return int(np.argmax(scores)), scores

print("Log-likelihood scoring function ready.")


In [ ]:
predictions = []
for i, item in enumerate(tqdm(test_ds, desc="Predicting")):
    pred_idx, _ = score_choices_loglik(model, processor, item["image"], item["prompt"], item["choices"])
    predictions.append({"id": item["id"], "answer": pred_idx})
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(test_ds)}] predictions generated")
print(f"\nDone. {len(predictions)} predictions generated.")


In [ ]:
submission_df = pd.DataFrame(predictions)
assert list(submission_df.columns) == ["id", "answer"]
assert pd.api.types.is_integer_dtype(submission_df["answer"])
assert len(submission_df) == len(test_df)
assert set(submission_df["id"]) == set(test_df["id"])
submission_path = Path("/kaggle/working/submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Saved: {submission_path} | Rows: {len(submission_df)}")
print("\nAnswer distribution:")
print(submission_df["answer"].value_counts().sort_index())
print("\nFirst 5 rows:")
print(submission_df.head())
